# CTC Validation Before GNW Integration

This notebook implements the proposed two-node S → W validation first. Stage 2 should use only parameter ranges that survive the controls here.

## What the supplied MATLAB actually contains

The file says its originally planned two-signal/two-layer architecture was not implemented. The active MATLAB branch (`determ = 1`) instead has one cortical E/I pair at V1 and one at V2. It uses noisy local E/I dynamics, a fixed one-way `S1E2E = 0.5` V1E→V2E projection, and nominal 10 Hz sine inputs with samplewise phase jitter that enter each inhibitory population (`alpha1`/`alpha2` times `alphafact = 3`). The E/I time constants, weights, and noise scale are copied into the scaffold. The alternative `determ = 0` branch includes pulvinar E/I quasi-cycles, but it is not the active default.

The MATLAB does **not** implement A/B/C controls, Kuramoto coupling, synchrony-dependent long-range weights, or GNW ignition. These are extensions in this notebook. The explicit `R`-gated V1→V2 weight is optional and separate, so it is not confused with the source's fixed projection plus alpha-to-inhibition mechanism.

The source sets `V2Eoff = 1` in executable code (one 50 μs step), although a nearby comment describes 125 steps (6.25 ms). The scaffold defaults to the executed value. MATLAB state variables are activity-like, **not rates in Hz**. Stage 1 therefore measures sustained W response calibrated from no-signal controls; do not call it the project's 20 Hz GNW ignition criterion.

## Reproduction status: this is not yet a numerical reproduction

The current Python code borrows the active branch's cortical E/I coefficients and fixed V1E→V2E projection, but its outputs should **not** be treated as reproductions of the MATLAB results. The Python default runs 1 s with a 200–600 ms stimulus; MATLAB runs 5 s and stimulates from about 2.5–5 s. More substantially, MATLAB drives inhibition with `sin(2π·10·t + 0.4·randn)` at each sample, while this scaffold uses explicit noisy phase oscillators and A/B/C coupling. The phase coherence is also computed at a different sampling/window scale, and the Python MI estimator is a quantile-binned estimate with shuffle-bias correction, unlike the MATLAB `mi(...)` routine.

So the current notebook tests an **extended CTC hypothesis inspired by the MATLAB**, not whether it reproduces the paper script's figures or numerical values. A true reproduction needs a separate source-faithful mode with the MATLAB time grid, alpha generation, analysis windows, and matching MI implementation.

## Step 1 — imports and reproducible setup

Default duration is 1 s for iteration; the supplied MATLAB used 5 s and applied its signal from 2.5 to 5 s.

In [ ]:
project_root = Path.cwd()
if (project_root / 'project_experiment' / 'ctc_validation.py').exists():
    experiment_dir = project_root / 'project_experiment'
elif (project_root / 'ctc_validation.py').exists():
    experiment_dir = project_root
else:  # launched with final_notebooks/ as the working directory
    project_root = project_root.parent
    experiment_dir = project_root / 'project_experiment'
sys.path.insert(0, str(experiment_dir))
from ctc_validation import DEFAULT_PARAMS, simulate_two_node_batch, calibrate_response_threshold

params = dict(DEFAULT_PARAMS)
print(f"dt={params['dt_s']*1e6:.0f} µs; run={params['duration_s']*1000:.0f} ms")
print(f"stimulus={params['stim_on_s']*1000:.0f}–{params['stim_off_s']*1000:.0f} ms")
print(f"V1→V2 delay={params['delay_samples']} step(s) = {params['delay_samples']*params['dt_s']*1000:g} ms")

## Source-faithful reproduction mode — MATLAB `determ = 1`

This separate implementation uses the source defaults: 100,000 steps at 50 μs (5 s), 10 runs, signal 20 from MATLAB index 50,001 to the end, independently jittered 10 Hz inputs, and the fixed delayed V1E→V2E projection. It computes the original raw phase coherence and the downsampled filtered coherence. The source's `mi.m` helper is absent from the supplied material, so MI below is explicitly a quantile-binned **proxy**, not a matched reproduction. A seeded NumPy run is reproducible within Python, but cannot duplicate the source's MATLAB random stream.

Start with one run if you only want to confirm the source-faithful code executes; use the defaults to compare summary distributions.

In [ ]:
from matlab_reference import MATLAB_ACTIVE_DEFAULTS, simulate_matlab_active_branch

# Full source settings: 5 s, 10 realizations. Override nits=1 for a quick execution check.
reference = simulate_matlab_active_branch(seed=2026, keep_traces=True)
reference_summary = pd.DataFrame({
    'raw_phase_PLV': reference['coherence_raw_phase'],
    'alpha_input_PLV': reference['coherence_alpha_input'],
    'filtered_10_20_Hz_PLV': reference['coherence_filtered_10_20_hz'],
    'filtered_PLV_source_scaled': reference['coherence_filtered_10_20_hz_source_scaled'],
    'signal_effect_peak_V2_minus_V1': reference['raw_signal_effect_peak_v2_minus_v1'],
    'signal_effect_near_peak_V2_minus_V1': reference['raw_signal_effect_near_peak_v2_minus_v1'],
})
display(reference_summary.describe().T)
print(reference['mi_comparability_note'])
print(reference['rng_note'])

In [ ]:
# Example visualization of one source-faithful realization.
ref_t_ms = reference['traces']['time_s'] * 1000
plt.figure(figsize=(9, 4))
plt.plot(ref_t_ms, reference['traces']['v1e'][0], label='V1E')
plt.plot(ref_t_ms, reference['traces']['v2e'][0], label='V2E', alpha=.8)
plt.axvspan(reference['stim_on_s']*1000, ref_t_ms[-1], color='gray', alpha=.18, label='Signal on')
plt.xlabel('Time (ms)'); plt.ylabel('Model activity (source units)')
plt.title('Source-faithful MATLAB active branch: V1→V2')
plt.grid(alpha=.25); plt.legend(); plt.tight_layout(); plt.show()

## Step 2 — define the controls

- **A:** mutually coupled phases and the V1E→V2E signal path.
- **B:** shared external 10 Hz rhythm but no V1E→V2E path. This tests whether common drive alone produces apparent synchrony.
- **C:** W phase is dragged one-way by S phase; the E projection remains on.

The MATLAB itself has a fixed one-way projection and alpha input to inhibition. A/B/C phase dynamics are experimental extensions, not three conditions from the source paper.

In [ ]:
configurations = ['A', 'B', 'C']
coupling_grid = [0, 5, 10, 15, 20, 30]  # rad/s, exploratory
paired_seeds = np.arange(1, 51)
signal_magnitude = 20.0  # MATLAB sigmag; verify pilot is not at ceiling
print(configurations, coupling_grid, f'{len(paired_seeds)} paired seeds')

## Step 3 — calibrate sustained-response threshold

Run a no-signal common-drive control. The threshold is based on the 99.5th percentile of 50 ms rolling-mean W activity after stimulus-off. This rejects isolated peaks. It is an operational activity-unit criterion, not a biological Hz threshold.

In [ ]:
calibration_seeds = np.arange(1001, 1021)
response_threshold, calibration = calibrate_response_threshold(
    calibration_seeds, params=params, percentile=99.5
)
print(f'W response threshold = {response_threshold:.4g} activity units')
print('This is not the GNW 20 Hz ignition criterion.')

## Step 4 — pilot before the full scan

Use a few paired seeds and representative coupling values. If responses are absent in every condition or guaranteed in every condition, revise `signal_magnitude` before the 50-seed scan. Keep the selected value fixed for confirmation.

In [ ]:
pilot_seeds = np.arange(201, 206)
pilot_rows = []
for cfg in configurations:
    for kappa in (0.0, 15.0, 30.0):
        out = simulate_two_node_batch(cfg, kappa, pilot_seeds, signal_magnitude,
                                      response_threshold=response_threshold, params=params)
        pilot_rows.append({
            'configuration': cfg, 'K_rad_s': kappa,
            'response_fraction': out['response_success'].mean(),
            'mean_R_alpha': out['mean_r_alpha_during'].mean(),
            'MI_corrected_bits': out['mi_activity_during_corrected_bits'].mean(),
            'mean_W_activity': out['mean_r_w_during'].mean(),
        })
display(pd.DataFrame(pilot_rows))

## Step 5 — Stage 1.1 paired-seed scan

Three configurations × six coupling values × 50 shared seeds. It records alpha PLV, activity-phase coherence, activity MI with circular-shift bias correction, W response probability, and response latency. First run the five-seed quick pass; then use all paired seeds. The full scan may take a few minutes.

In [ ]:
def run_stage1(seeds, couplings=coupling_grid, *, explicit_weight_gate=False):
    rows = []
    for cfg in configurations:
        for kappa in couplings:
            out = simulate_two_node_batch(
                cfg, kappa, seeds, signal_magnitude,
                explicit_weight_gate=explicit_weight_gate,
                response_threshold=response_threshold, params=params)
            for i, seed in enumerate(seeds):
                rows.append({
                    'configuration': cfg, 'K_rad_s': kappa, 'seed': int(seed),
                    'response_success': bool(out['response_success'][i]),
                    'response_latency_ms': out['response_latency_ms'][i],
                    'R_alpha': out['mean_r_alpha_during'][i],
                    'R_gamma_activity': out['mean_r_gamma_during'][i],
                    'MI_activity_bits': out['mi_activity_during_bits'][i],
                    'MI_corrected_bits': out['mi_activity_during_corrected_bits'][i],
                    'mean_W_activity': out['mean_r_w_during'][i],
                    'max_W_activity': out['max_r_w'][i],
                    'explicit_weight_gate': explicit_weight_gate,
                })
    return pd.DataFrame(rows)

# Quick check; after inspecting, replace [:5] with all paired_seeds.
stage1_pilot = run_stage1(paired_seeds[:5])
display(stage1_pilot.groupby(['configuration', 'K_rad_s']).agg(
    response_probability=('response_success', 'mean'),
    mean_R_alpha=('R_alpha', 'mean'),
    mean_MI_corrected=('MI_corrected_bits', 'mean')).reset_index())

## Step 6 — full scan and summary

Keep the per-seed table for paired uncertainty estimates. Review the pilot before launching the full scan.

In [ ]:
# Uncomment after reviewing the pilot.
# stage1 = run_stage1(paired_seeds)
# summary = stage1.groupby(['configuration', 'K_rad_s']).agg(
#     response_probability=('response_success', 'mean'),
#     mean_latency_ms=('response_latency_ms', 'mean'),
#     mean_R_alpha=('R_alpha', 'mean'),
#     mean_R_gamma_activity=('R_gamma_activity', 'mean'),
#     mean_MI_corrected_bits=('MI_corrected_bits', 'mean'),
#     mean_max_W_activity=('max_W_activity', 'mean'),
# ).reset_index()
# display(summary)
# stage1.to_csv(project_root / 'ctc_stage1_paired_seed_results.csv', index=False)

## Step 7 — does synchrony predict useful communication?

Compare `R_alpha`, corrected MI, and response probability across A/B/C. A high PLV in B warns that common drive can create synchrony without a communication path. C tests one-way phase dragging. MI is an estimator requiring bias checks, not causal proof.

At the same or overlapping R values, ask whether A yields more W responses or corrected MI than B/C. If R ranges do not overlap, that comparison is not identified; add conditions to create overlap rather than extrapolating.

In [ ]:
# Run after the full scan.
# fig, axes = plt.subplots(1, 2, figsize=(11, 4))
# for cfg, group in stage1.groupby('configuration'):
#     means = group.groupby('K_rad_s').agg(R=('R_alpha','mean'), success=('response_success','mean'))
#     axes[0].plot(means.R, means.success, marker='o', label=cfg)
#     axes[1].plot(means.index, group.groupby('K_rad_s').MI_corrected_bits.mean(), marker='o', label=cfg)
# axes[0].set(xlabel='Mean alpha PLV during stimulus', ylabel='W response probability', ylim=(-.05,1.05))
# axes[1].set(xlabel='K (rad/s)', ylabel='Bias-corrected activity MI (bits)')
# for ax in axes: ax.grid(alpha=.25); ax.legend(title='Configuration')
# plt.tight_layout(); plt.show()

## Step 8 — optional explicit synchrony-gated projection

Only after understanding A/B/C, rerun selected conditions with `explicit_weight_gate=True`. This multiplies fixed V1E→V2E coupling by a synchrony/phase term, which is a new hypothesis beyond the MATLAB. Compare to identical gate-off conditions and keep the flag in every result table.

## Stage 2 decision gate

Proceed to the 40-area GNW analysis only if the communication advantage repeats across paired seeds, is not explained by common-drive synchrony (B), is not reproduced by one-way dragging alone (C), and does not reflect pathological activity. Stage 2 must use the original model's actual Hz rates and its own ignition criterion. If Stage 1 does not pass, report that this parameter range did not establish CTC-mediated communication rather than tuning the threshold until it passes.